
|  **Hidden Gems Scout** | Find young (≤21), undervalued, low-minute players with high potential |
|  **Player Similarity Finder** | Find the most statistically similar players to any chosen player |

---

## 0 · Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.patches import FancyBboxPatch

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'font.family':      'DejaVu Sans',
})

ACCENT   = '#58a6ff'   # blue
GOLD     = '#d4a017'   # gold / hidden gem colour
GREEN    = '#3fb950'   # positive
RED      = '#f85149'   # negative
PURPLE   = '#bc8cff'   # similarity

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# ── Data paths ─────────────────────────────────────────────────────────────
# Adjust if your files live elsewhere
DATA_DIR = '../data/clean'

import os
# Fallback: look in the current directory
if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'

# ── Load outfield master ranking ───────────────────────────────────────────
master_path = os.path.join(DATA_DIR, 'master_player_ranking.csv')
df_master   = pd.read_csv(master_path)

# ── Load goalkeepers ───────────────────────────────────────────────────────
gk_path = os.path.join(DATA_DIR, 'classified_goalkeepers.csv')
df_gk   = pd.read_csv(gk_path)

# Harmonise GK column names so we can stack them if needed
if 'Broad_Position' in df_gk.columns:
    df_gk = df_gk.rename(columns={'Broad_Position': 'Position'})

# Remove any duplicate columns (GK file can have duplicate 'Position')
df_gk     = df_gk.loc[:, ~df_gk.columns.duplicated()]
df_master = df_master.loc[:, ~df_master.columns.duplicated()]

# Combine for the similarity engine (outfield + GK together)
common_cols = sorted(set(df_master.columns.tolist()) & set(df_gk.columns.tolist()))
df_all = pd.concat([df_master[common_cols], df_gk[common_cols]], ignore_index=True)

# ── Pretty-print league labels ─────────────────────────────────────────────
LEAGUE_MAP = {
    'premier-league': 'Premier League',
    'la-liga':        'La Liga',
    'bundesliga':     'Bundesliga',
    'serie-a':        'Serie A',
    'ligue-1':        'Ligue 1',
}
if 'League' in df_master.columns:
    df_master['League_Label'] = df_master['League'].map(LEAGUE_MAP).fillna(df_master['League'])
    df_all['League_Label']    = df_all['League'].map(LEAGUE_MAP).fillna(df_all['League'])

print(f' Outfield players loaded : {len(df_master):,}')
print(f' Goalkeepers loaded      : {len(df_gk):,}')
print(f' Combined dataset        : {len(df_all):,}')
print(f' Leagues                 : {sorted(df_master["League"].unique()) if "League" in df_master.columns else "N/A"}')
print(f' Positions               : {sorted(df_master["Position"].unique())}')

 Outfield players loaded : 772
 Goalkeepers loaded      : 52
 Combined dataset        : 824
 Leagues                 : ['la-liga', 'premier-league']
 Positions               : ['Attacking Midfield', 'Central Midfield', 'Centre-Back', 'Defensive Midfield', 'Full-Back', 'Striker', 'Winger']


---
##  Part 1 — Hidden Gems Scout

**Logic:** A _hidden gem_ is a young player who is:
- **Young** — Age ≤ threshold (default 21)
- **Cheap** — Market Value ≤ _X_-th percentile of players at the same position
- **Under-used** — Minutes Played ≤ threshold (e.g. < 900 min = < 10 full games)
- **Talented** — Performance_Score ≥ _Y_-th percentile among eligible peers

A **Gem Score** is computed to rank them: high performance + low price + low age.

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER — compute gem score and return filtered dataframe
# ══════════════════════════════════════════════════════════════════════════

def find_hidden_gems(
    df,
    max_age          = 21,
    max_minutes      = 1350,
    mv_percentile    = 40,     # top % market value cutoff within position group
    min_perf_pct     = 0,      # minimum Performance_Score percentile to include
    positions        = None,   # None = all positions
    leagues          = None,   # None = all leagues
    top_n            = 20,
):
    """
    Returns a ranked dataframe of hidden gem candidates.
    """
    d = df.copy()

    # ── Position & League filters ──────────────────────────────────────────
    if positions and 'Position' in d.columns:
        d = d[d['Position'].isin(positions)]
    if leagues and 'League' in d.columns:
        d = d[d['League'].isin(leagues)]

    # ── Core criteria ──────────────────────────────────────────────────────
    d['Age']          = pd.to_numeric(d['Age'],          errors='coerce')
    d['Minutes']      = pd.to_numeric(d['Minutes'],      errors='coerce')
    d['Market Value'] = pd.to_numeric(d['Market Value'], errors='coerce')

    d = d[d['Age'].notna() & (d['Age'] <= max_age)]
    d = d[d['Minutes'].notna() & (d['Minutes'] <= max_minutes)]
    d = d[d['Market Value'].notna()]

    if d.empty:
        print('  No players found with these criteria. Try relaxing the filters.')
        return pd.DataFrame()

    # Market-value ceiling per position group
    def mv_cutoff(grp):
        cutoff = np.percentile(grp['Market Value'], mv_percentile)
        return grp[grp['Market Value'] <= cutoff]

    if 'Position' in d.columns:
        d = d.groupby('Position', group_keys=False).apply(mv_cutoff)
    else:
        cutoff = np.percentile(d['Market Value'], mv_percentile)
        d = d[d['Market Value'] <= cutoff]

    # Minimum performance filter
    if 'Performance_Score' in d.columns and not d.empty:
        d['Performance_Score'] = pd.to_numeric(d['Performance_Score'], errors='coerce')
        perf_floor = np.percentile(d['Performance_Score'].dropna(), min_perf_pct)
        d = d[d['Performance_Score'] >= perf_floor]

    if d.empty:
        print('  No players survived all filters.')
        return pd.DataFrame()

    # ── Gem Score ──────────────────────────────────────────────────────────
    # Normalise 0-1 within the filtered pool
    scaler = MinMaxScaler()

    def norm(series, invert=False):
        vals = series.fillna(series.median()).values.reshape(-1, 1)
        scaled = scaler.fit_transform(vals).flatten()
        return 1 - scaled if invert else scaled

    d = d.copy()
    d['_perf_norm'] = norm(d['Performance_Score']) if 'Performance_Score' in d.columns else 0.5
    d['_mv_norm']   = norm(d['Market Value'], invert=True)   # cheaper = higher
    d['_age_norm']  = norm(d['Age'],          invert=True)   # younger = higher

    # Weighted gem score: 50% perf, 30% value-for-money, 20% youth
    d['Gem_Score'] = (d['_perf_norm'] * 50 +
                      d['_mv_norm']   * 30 +
                      d['_age_norm']  * 20)

    d = d.sort_values('Gem_Score', ascending=False).reset_index(drop=True)
    d.index = d.index + 1          # 1-based rank
    d.index.name = 'Gem_Rank'

    # ── Select output columns ─────────────────────────────────────────────
    base_cols = ['Name', 'Team', 'Age', 'Position', 'Tactical_Role',
                 'League', 'Minutes', 'Market Value', 'Performance_Score', 'Gem_Score']
    out_cols = [c for c in base_cols if c in d.columns]
    return d[out_cols].head(top_n)


print(' find_hidden_gems() defined.')

 find_hidden_gems() defined.


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER — visualise the gems
# ══════════════════════════════════════════════════════════════════════════

def plot_gems(gems_df, title='Hidden Gems'):
    if gems_df.empty:
        return

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle(f'  {title}', fontsize=17, fontweight='bold', color=GOLD, y=1.01)

    # ── Plot 1: Gem Score bar ─────────────────────────────────────────────
    ax = axes[0]
    top10 = gems_df.head(10)
    colors = [GOLD if i == 0 else ACCENT for i in range(len(top10))]
    bars = ax.barh(top10['Name'][::-1], top10['Gem_Score'][::-1],
                   color=colors[::-1], edgecolor='none', height=0.65)
    ax.set_xlabel('Gem Score', fontsize=11)
    ax.set_title('Top 10 by Gem Score', fontsize=13, color=GOLD)
    ax.set_xlim(0, top10['Gem_Score'].max() * 1.15)
    for bar, val in zip(bars, top10['Gem_Score'][::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9, color='#e6edf3')
    ax.grid(axis='x', alpha=0.3)
    ax.set_facecolor('#161b22')

    # ── Plot 2: Market Value vs Performance_Score scatter ─────────────────
    ax2 = axes[1]
    if 'Performance_Score' in gems_df.columns and 'Market Value' in gems_df.columns:
        mv_m = gems_df['Market Value'] / 1e6   # in millions
        sc = ax2.scatter(mv_m, gems_df['Performance_Score'],
                         c=gems_df['Gem_Score'], cmap='YlOrRd',
                         s=100, edgecolors='#30363d', linewidth=0.5, zorder=3)
        # Label top 5
        for _, row in gems_df.head(5).iterrows():
            ax2.annotate(row['Name'].split()[-1],
                         xy=(row['Market Value']/1e6, row['Performance_Score']),
                         xytext=(4, 4), textcoords='offset points',
                         fontsize=8, color=GOLD)
        cb = plt.colorbar(sc, ax=ax2)
        cb.set_label('Gem Score', color='#8b949e')
        cb.ax.yaxis.set_tick_params(color='#8b949e')
        plt.setp(cb.ax.yaxis.get_ticklabels(), color='#8b949e')
    ax2.set_xlabel('Market Value (€M)', fontsize=11)
    ax2.set_ylabel('Performance Score', fontsize=11)
    ax2.set_title('Value vs Performance', fontsize=13, color=GOLD)
    ax2.grid(alpha=0.3)
    ax2.set_facecolor('#161b22')

    # ── Plot 3: Age distribution ──────────────────────────────────────────
    ax3 = axes[2]
    age_counts = gems_df['Age'].value_counts().sort_index()
    ax3.bar(age_counts.index.astype(int), age_counts.values,
            color=ACCENT, edgecolor='none', width=0.6)
    ax3.set_xlabel('Age', fontsize=11)
    ax3.set_ylabel('# Players', fontsize=11)
    ax3.set_title('Age Distribution of Gems', fontsize=13, color=GOLD)
    ax3.grid(axis='y', alpha=0.3)
    ax3.set_facecolor('#161b22')

    plt.tight_layout()
    plt.show()


print(' plot_gems() defined.')

 plot_gems() defined.


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# INTERACTIVE WIDGET — Hidden Gems Scout
# ══════════════════════════════════════════════════════════════════════════

all_positions = sorted(df_master['Position'].dropna().unique().tolist()) if 'Position' in df_master.columns else []
all_leagues   = sorted(df_master['League'].dropna().unique().tolist())   if 'League'   in df_master.columns else []

style     = {'description_width': '170px'}
layout_w  = widgets.Layout(width='420px')
layout_s  = widgets.Layout(width='420px', height='130px')

w_age     = widgets.IntSlider(value=21, min=16, max=25, step=1,
                               description='Max Age:', style=style, layout=layout_w)
w_minutes = widgets.IntSlider(value=1350, min=90, max=2700, step=90,
                               description='Max Minutes:', style=style, layout=layout_w)
w_mv_pct  = widgets.IntSlider(value=40, min=5, max=90, step=5,
                               description='MV Percentile Cutoff:', style=style, layout=layout_w)
w_perf    = widgets.IntSlider(value=0, min=0, max=80, step=5,
                               description='Min Perf Score %ile:', style=style, layout=layout_w)
w_top_n   = widgets.IntSlider(value=20, min=5, max=50, step=5,
                               description='Top N Results:', style=style, layout=layout_w)
w_pos     = widgets.SelectMultiple(options=['All Positions'] + all_positions,
                                    value=['All Positions'],
                                    description='Positions:', style=style, layout=layout_s)
w_league  = widgets.SelectMultiple(options=['All Leagues'] + all_leagues,
                                    value=['All Leagues'],
                                    description='Leagues:', style=style, layout=layout_s)
w_btn     = widgets.Button(description='  Find Hidden Gems',
                            button_style='warning',
                            layout=widgets.Layout(width='220px', height='38px'))
gem_out   = widgets.Output()

def on_gem_click(_):
    with gem_out:
        clear_output(wait=True)
        pos_sel = None if 'All Positions' in w_pos.value else list(w_pos.value)
        lg_sel  = None if 'All Leagues'   in w_league.value else list(w_league.value)
        gems    = find_hidden_gems(
            df_master,
            max_age       = w_age.value,
            max_minutes   = w_minutes.value,
            mv_percentile = w_mv_pct.value,
            min_perf_pct  = w_perf.value,
            positions     = pos_sel,
            leagues       = lg_sel,
            top_n         = w_top_n.value,
        )
        if not gems.empty:
            # Format market value column for display
            display_df = gems.copy()
            if 'Market Value' in display_df.columns:
                display_df['Market Value'] = display_df['Market Value'].apply(
                    lambda x: f'€{x/1e6:.1f}M' if pd.notna(x) else 'N/A')
            if 'Gem_Score' in display_df.columns:
                display_df['Gem_Score'] = display_df['Gem_Score'].round(2)
            if 'Performance_Score' in display_df.columns:
                display_df['Performance_Score'] = display_df['Performance_Score'].round(2)
            display(HTML('<h3 style="color:#d4a017"> Hidden Gem Results</h3>'))
            display(display_df.style
                    .background_gradient(subset=['Gem_Score'], cmap='YlOrRd')
                    .set_properties(**{'text-align': 'left'})
                    .format({'Gem_Score': '{:.2f}', 'Performance_Score': '{:.2f}'}, na_rep='—'))
            plot_gems(gems, title=f'Hidden Gems — Age ≤ {w_age.value}, Minutes ≤ {w_minutes.value}')

w_btn.on_click(on_gem_click)

# ── Layout ────────────────────────────────────────────────────────────────
col1 = widgets.VBox([w_age, w_minutes, w_mv_pct, w_perf, w_top_n])
col2 = widgets.VBox([w_pos, w_league])
panel = widgets.VBox([
    widgets.HTML('<h2 style="color:#d4a017; font-family:monospace"> Hidden Gems Scout</h2>'),
    widgets.HBox([col1, col2]),
    w_btn,
    gem_out
])
display(panel)

---
##  Part 2 — Player Similarity Finder

**Logic:** Given a reference player, we compute _cosine similarity_ across all **per-90-minute statistics** (so playing time doesn't inflate the scores). The result is ranked from most to least similar.

Filters can be applied to narrow the candidate pool (e.g. _find the most similar striker playing in La Liga_).

A **radar chart** is also drawn to compare the reference player against the top match.

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# BUILD FEATURE MATRIX FOR SIMILARITY
# ══════════════════════════════════════════════════════════════════════════

# Use per-90 columns + a few rate stats — position-agnostic features
PER90_COLS = [c for c in df_all.columns if 'Per90' in c]
RATE_COLS  = [c for c in df_all.columns
              if any(k in c for k in ['Rate', 'Accuracy', 'Percentage'])
              and 'Per90' not in c]

FEATURE_COLS = PER90_COLS + RATE_COLS
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_all.columns]

print(f'Feature columns used for similarity ({len(FEATURE_COLS)}):')
print(', '.join(FEATURE_COLS))

Feature columns used for similarity (25):
Aerial Duels Won (Per90), Assists (Per90), Clearances (Per90), Crosses (Per90), Dispossesed (Per90), Dribbled Past (Per90), Dribbles (Per90), Expected Assists (xA) (Per90), Fouled Against (Per90), Fouls Committed (Per90), Ground Duels (Per90), Ground Duels Won (Per90), Interceptions (Per90), Key Passes (Per90), Offsides (Per90), Passes (Per90), Penalties Given Away (Per90), Successful Crosses (Per90), Successful Dribbles (Per90), Successful Passes (Per90), Tackles (Per90), Total Cards (Per90), Cross Completion Rate, Dribble Success Rate, Pass Completion Rate


In [7]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER — find similar players
# ══════════════════════════════════════════════════════════════════════════

def find_similar_players(
    df,
    player_name,
    top_n          = 10,
    same_position  = False,
    positions      = None,
    leagues        = None,
    max_age        = None,
    min_minutes    = None,
    feature_cols   = None,
):
    """
    Returns (ref_row, similar_df) — the reference player and the top-N matches.
    """
    if feature_cols is None:
        feature_cols = FEATURE_COLS

    # ── Find reference player ─────────────────────────────────────────────
    matches = df[df['Name'].str.lower() == player_name.strip().lower()]
    if matches.empty:
        # Fuzzy fallback: contains
        matches = df[df['Name'].str.lower().str.contains(player_name.strip().lower(), na=False)]
    if matches.empty:
        print(f' Player "{player_name}" not found in dataset.')
        return None, pd.DataFrame()

    ref = matches.iloc[0]
    ref_pos = ref.get('Position', None)

    # ── Build candidate pool ──────────────────────────────────────────────
    pool = df[df['Name'] != ref['Name']].copy()

    if same_position and ref_pos:
        pool = pool[pool['Position'] == ref_pos]
    if positions:
        pool = pool[pool['Position'].isin(positions)]
    if leagues and 'League' in pool.columns:
        pool = pool[pool['League'].isin(leagues)]
    if max_age and 'Age' in pool.columns:
        pool = pool[pd.to_numeric(pool['Age'], errors='coerce') <= max_age]
    if min_minutes and 'Minutes' in pool.columns:
        pool = pool[pd.to_numeric(pool['Minutes'], errors='coerce') >= min_minutes]

    if pool.empty:
        print('  No candidates after filters.')
        return ref, pd.DataFrame()

    # ── Compute cosine similarity ─────────────────────────────────────────
    available = [c for c in feature_cols if c in pool.columns]
    all_rows  = pd.concat([matches.head(1), pool], ignore_index=True)
    X = all_rows[available].apply(pd.to_numeric, errors='coerce').fillna(0)

    scaler   = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Row 0 is the reference player
    sim_scores = cosine_similarity([X_scaled[0]], X_scaled[1:])[0]
    pool = pool.copy()
    pool['Similarity_%'] = (sim_scores * 100).round(2)
    pool = pool.sort_values('Similarity_%', ascending=False).reset_index(drop=True)
    pool.index = pool.index + 1
    pool.index.name = 'Rank'

    out_cols = ['Name', 'Team', 'Age', 'Position', 'Tactical_Role',
                'League', 'Minutes', 'Market Value', 'Performance_Score', 'Similarity_%']
    out_cols = [c for c in out_cols if c in pool.columns]
    return ref, pool[out_cols].head(top_n)


print(' find_similar_players() defined.')

 find_similar_players() defined.


In [8]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER — Radar chart comparison
# ══════════════════════════════════════════════════════════════════════════

# Radar dimensions (8 key per-90 stats)
RADAR_METRICS = [
    'Goals Scored (Per90)',
    'Assists (Per90)',
    'Key Passes (Per90)',
    'Tackles (Per90)',
    'Interceptions (Per90)',
    'Successful Dribbles (Per90)',
    'Shots On Target (Per90)',
    'Pass Completion Rate',
]

def radar_compare(df, ref_row, similar_row, title='Player Comparison'):
    """
    Draw a radar chart comparing two players.
    """
    available = [m for m in RADAR_METRICS if m in df.columns]
    if len(available) < 3:
        print('  Not enough metrics for radar chart.')
        return

    # Normalise against the full dataset
    pool_vals = df[available].apply(pd.to_numeric, errors='coerce').fillna(0)
    scaler    = MinMaxScaler()
    scaler.fit(pool_vals)

    def get_scaled(row):
        vals = [pd.to_numeric(row.get(m, 0), errors='coerce') or 0 for m in available]
        return scaler.transform([vals])[0]

    ref_scaled  = get_scaled(ref_row)
    sim_scaled  = get_scaled(similar_row)

    N      = len(available)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]   # close the loop

    ref_vals = list(ref_scaled) + [ref_scaled[0]]
    sim_vals = list(sim_scaled) + [sim_scaled[0]]

    short_labels = [m.replace(' (Per90)', '').replace('Pass Completion Rate', 'Pass %') for m in available]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#161b22')

    # Grid lines
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['', '', '', '', ''], color='#8b949e', size=7)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(short_labels, color='#e6edf3', size=10)
    ax.tick_params(axis='x', pad=14)
    ax.spines['polar'].set_color('#30363d')
    ax.grid(color='#30363d', linewidth=0.8)

    # Reference player
    ref_name = ref_row.get('Name', 'Reference')
    ax.plot(angles, ref_vals, color=ACCENT, linewidth=2.5, label=ref_name)
    ax.fill(angles, ref_vals, color=ACCENT, alpha=0.15)

    # Similar player
    sim_name = similar_row.get('Name', 'Similar')
    ax.plot(angles, sim_vals, color=PURPLE, linewidth=2.5, linestyle='--', label=sim_name)
    ax.fill(angles, sim_vals, color=PURPLE, alpha=0.10)

    sim_pct = similar_row.get('Similarity_%', '')
    ax.set_title(f'{title}\n{ref_name} vs {sim_name}  ({sim_pct}% match)',
                 fontsize=13, fontweight='bold', color='#e6edf3', pad=22)
    ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.12),
              facecolor='#161b22', edgecolor='#30363d', labelcolor='#e6edf3')

    plt.tight_layout()
    plt.show()


print(' radar_compare() defined.')

 radar_compare() defined.


In [9]:
# ══════════════════════════════════════════════════════════════════════════
# INTERACTIVE WIDGET — Player Similarity Finder
# ══════════════════════════════════════════════════════════════════════════

all_names_combined = sorted(df_all['Name'].dropna().unique().tolist())

style2    = {'description_width': '170px'}
layout_w2 = widgets.Layout(width='420px')
layout_s2 = widgets.Layout(width='420px', height='130px')

# Autocomplete player name
w_player    = widgets.Combobox(
                  options=all_names_combined,
                  description='Player Name:',
                  placeholder='Start typing a name…',
                  ensure_option=False, style=style2, layout=layout_w2)

w_top_n2    = widgets.IntSlider(value=10, min=3, max=30, step=1,
                                 description='Top N Similar:', style=style2, layout=layout_w2)



w_max_age2  = widgets.IntSlider(value=45, min=16, max=45, step=1,
                                 description='Max Age:', style=style2, layout=layout_w2)

w_min_min   = widgets.IntSlider(value=0, min=0, max=2000, step=90,
                                 description='Min Minutes:', style=style2, layout=layout_w2)

w_pos2      = widgets.SelectMultiple(
                  options=['All Positions'] + all_positions,
                  value=['All Positions'],
                  description='Filter Positions:', style=style2, layout=layout_s2)

w_league2   = widgets.SelectMultiple(
                  options=['All Leagues'] + all_leagues,
                  value=['All Leagues'],
                  description='Filter Leagues:', style=style2, layout=layout_s2)

w_radar     = widgets.Checkbox(value=True, description='Show radar chart vs #1 match',
                                style=style2, layout=layout_w2)

w_btn2      = widgets.Button(description='  Find Similar Players',
                              button_style='info',
                              layout=widgets.Layout(width='240px', height='38px'))
sim_out     = widgets.Output()

def on_sim_click(_):
    with sim_out:
        clear_output(wait=True)
        pos_sel = None if 'All Positions' in w_pos2.value else list(w_pos2.value)
        lg_sel  = None if 'All Leagues'   in w_league2.value else list(w_league2.value)
        max_a   = w_max_age2.value if w_max_age2.value < 45 else None
        min_m   = w_min_min.value  if w_min_min.value  > 0  else None

        ref_row, sim_df = find_similar_players(
            df_all,
            player_name   = w_player.value,
            top_n         = w_top_n2.value,
            same_position = True,
            positions     = pos_sel,
            leagues       = lg_sel,
            max_age       = max_a,
            min_minutes   = min_m,
        )

        if ref_row is None or sim_df.empty:
            return

        # Reference player summary
        display(HTML(f"""
            <div style="background:#161b22;border:1px solid #30363d;
                        border-radius:8px;padding:12px;margin-bottom:10px;">
                <h3 style="color:#58a6ff;margin:0">🎯 Reference Player</h3>
                <b style="color:#e6edf3">{ref_row.get('Name','')}</b>
                &nbsp;|&nbsp; {ref_row.get('Team','')}
                &nbsp;|&nbsp; Age {int(ref_row.get('Age',0))}
                &nbsp;|&nbsp; {ref_row.get('Position','')}
                &nbsp;|&nbsp; {ref_row.get('Tactical_Role','')}
                &nbsp;|&nbsp; {str(ref_row.get('League','')).replace('-',' ').title()}
                &nbsp;|&nbsp; €{pd.to_numeric(ref_row.get('Market Value',0), errors='coerce')/1e6:.1f}M
            </div>
        """))

        # Similar players table
        display_df = sim_df.copy()
        if 'Market Value' in display_df.columns:
            display_df['Market Value'] = display_df['Market Value'].apply(
                lambda x: f'€{pd.to_numeric(x, errors="coerce")/1e6:.1f}M' if pd.notna(x) else 'N/A')
        display(HTML('<h3 style="color:#bc8cff"> Most Similar Players</h3>'))
        display(display_df.style
                .background_gradient(subset=['Similarity_%'], cmap='Blues')
                .set_properties(**{'text-align': 'left'})
                .format({'Similarity_%': '{:.2f}%', 'Performance_Score': '{:.2f}'}, na_rep='—'))

        # Radar chart vs top match
        if w_radar.value and not sim_df.empty:
            top_match = sim_df.iloc[0]
            # Merge top_match with df_all row for all stat columns
            full_match = df_all[df_all['Name'] == top_match.get('Name', '')]
            if not full_match.empty:
                radar_compare(df_all, ref_row, full_match.iloc[0])

w_btn2.on_click(on_sim_click)

col1b = widgets.VBox([w_player, w_top_n2, w_max_age2, w_min_min, w_radar])
col2b = widgets.VBox([w_pos2, w_league2])
panel2 = widgets.VBox([
    widgets.HTML('<h2 style="color:#58a6ff; font-family:monospace"> Player Similarity Finder</h2>'),
    widgets.HBox([col1b, col2b]),
    w_btn2,
    sim_out
])
display(panel2)

---
*Built on top of the player classification & ranking pipeline · `03_processing.ipynb`*